# 06 · PrimateAI — the semi-supervised alternative

**PrimateAI** (Sundaram et al. 2018, *Nat Genet*, PMID 30038395) is a deep net that dodges the need for curated pathogenic labels: it trains on a **proxy for benignity** — missense variants common in humans or seen in other primates. That makes it **semi-supervised**, with only **medium** circularity vs ClinVar.

> ✅ **REAL DATA (subset).** PrimateAI for CFTR from **dbNSFP v5.0a** — `data/primateai_cftr.csv`, built by a manual-download build cell below. ⚠️ **Coverage:** dbNSFP's ClinVar-re-annotated subset, so **~1,976 observed** CFTR variants (NOT saturation). Non-commercial. `source == 'REAL'`.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · PrimateAI — the *semi-supervised* alternative

**PrimateAI** (Sundaram et al. 2018, *Nature Genetics*, **PMID 30038395**) is a **deep neural
network**, but it dodges the "we need labelled pathogenic variants" problem with a neat trick.

Instead of training on curated *pathogenic* labels, it trains on a **proxy for benignity**:

> A missense variant that is **common** in healthy humans — or is seen as the normal amino acid in
> **other primates** (chimp, gorilla, …) — is very likely **tolerated / benign**. Evolution has
> already "tested" it.

So PrimateAI learns from **hundreds of thousands of common human & non-human-primate missense
variants** as its benign class. This is why it's called **semi-supervised**: it uses labels, but
those labels are a *population-frequency proxy*, **not** clinical pathogenic/benign assertions.

| Property | PrimateAI |
|---|---|
| Learning type | **Semi-supervised** (benign proxy from primate/common variants) |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Pathogenic cut | **≥ 0.803** |
| Circularity vs ClinVar | **Medium** — it never trained on ClinVar *pathogenic* labels, but common-variant proxies can still overlap benign ClinVar entries |


### Building the REAL data — a manual download (via dbNSFP, not PrimateAI's own release)

The original PrimateAI shipped precomputed genome-wide scores from Illumina
(BaseSpace/Zenodo), which are large and gated. Instead, this toolkit pulls
PrimateAI's score **out of dbNSFP v5.0a**, a separate database that bundles a
`PrimateAI_score` column for its **ClinVar-re-annotated variant subset** — much
smaller and easier to get, at the cost of coverage (not saturation; see below).
**You must fetch this one yourself:**

1. Get dbNSFP v5.0a's `dbNSFP5.0a_variant.clin_var_re-annot_pdb_variants_plddt_rsa.parquet`
   (Zenodo record 15131632).
2. Save it as `data/dbNSFP5.0a_variant.clin_var_re-annot_pdb_variants_plddt_rsa.parquet`
   (gitignored — never commit it).

The cell below reads only chromosome 7 from the parquet (column-pruned, so it
never loads the other chromosomes), filters to the CFTR GRCh38 window, keeps
missense rows, and derives the 1-letter `protein_variant` key.

License: dbNSFP is **CC BY-NC-ND** (non-commercial, no derivatives); PrimateAI
itself is also non-commercial — see `data_manifest.json`.

In [2]:
import pyarrow.parquet as pq

DATA_DIR = pathlib.Path.cwd().parent / "data"
DBNSFP_PARQUET = DATA_DIR / "dbNSFP5.0a_variant.clin_var_re-annot_pdb_variants_plddt_rsa.parquet"
PRIMATEAI_TSV = DATA_DIR / "primateai_cftr.csv"
CFTR_WINDOW = (117_470_000, 117_670_000)   # GRCh38
AA1 = set("ACDEFGHIKLMNPQRSTVWY")

if PRIMATEAI_TSV.exists():
    print(f"already built -> {PRIMATEAI_TSV.name} (delete to rebuild)")
elif not DBNSFP_PARQUET.exists():
    raise FileNotFoundError(
        f"{DBNSFP_PARQUET} not found.\n"
        "PrimateAI's own genome-wide release is large/gated -- we pull the score\n"
        "from dbNSFP v5.0a's ClinVar-re-annotated subset instead:\n"
        "  1. Get dbNSFP v5.0a (Zenodo record 15131632):\n"
        "     'dbNSFP5.0a_variant.clin_var_re-annot_pdb_variants_plddt_rsa.parquet'\n"
        f"  2. Save it as {DBNSFP_PARQUET} (do NOT commit it -- data/ is gitignored)\n"
        "Then re-run this cell -- it reads only chromosome 7's columns, never the whole file."
    )
else:
    start, end = CFTR_WINDOW
    cols = ["#chr", "pos(1-based)", "ref", "alt", "aaref", "aaalt", "aapos",
            "PrimateAI_score", "PrimateAI_pred"]
    df = pq.read_table(DBNSFP_PARQUET, columns=cols, filters=[("#chr", "==", "7")]).to_pandas()
    df["pos"] = df["pos(1-based)"].astype("int64")
    cf = df[(df["pos"] >= start) & (df["pos"] <= end)].copy()

    def num(s):
        try:
            return float(s)
        except (ValueError, TypeError):
            return None

    cf["primate_ai_score"] = cf["PrimateAI_score"].map(num)
    cf = cf.dropna(subset=["primate_ai_score"])
    cf = cf[cf["aaref"].isin(AA1) & cf["aaalt"].isin(AA1)]           # missense only
    cf["protein_variant"] = cf["aaref"] + cf["aapos"].astype(str) + cf["aaalt"]
    cf["chrom"] = cf["#chr"].astype(str)
    out = (cf[["chrom", "pos", "ref", "alt", "protein_variant", "primate_ai_score", "PrimateAI_pred"]]
           .rename(columns={"PrimateAI_pred": "primate_ai_pred"})
           .drop_duplicates("protein_variant").sort_values("pos").reset_index(drop=True))
    out["source"] = "REAL"
    out.to_csv(PRIMATEAI_TSV, index=False)
    print(f"REAL PrimateAI CFTR variants written: {len(out):,} -> {PRIMATEAI_TSV.relative_to(DATA_DIR.parent)}")

already built -> primateai_cftr.csv (delete to rebuild)


In [3]:
tk.THRESHOLDS['primate_ai']

{'path': 0.803, 'benign': 0.483}

## 2 · Load PrimateAI and make a call

Score 0-1, higher = worse, pathogenic cut `>= 0.803`.

In [4]:
primateai = tk.load_primateai()   # REAL — dbNSFP subset (~1,976 observed CFTR variants), built above
print(f"{len(primateai):,} REAL PrimateAI variants | source: {primateai['source'].unique().tolist()}")
print('score range:', primateai['primate_ai_score'].min(), '->', primateai['primate_ai_score'].max(),
      '| pathogenic (>= 0.803):', int((primateai['primate_ai_score'] >= 0.803).sum()))
primateai['pai_call'] = primateai['primate_ai_score'].apply(lambda s: tk.call_from_score(s, 'primate_ai'))
primateai[['protein_variant', 'primate_ai_score', 'pai_call', 'source']].head(10)

1,976 REAL PrimateAI variants | source: ['REAL']
score range: 0.176448047161 -> 0.932814955711 | pathogenic (>= 0.803): 132


,protein_variant,primate_ai_score,pai_call,source
0,Q2P,0.612750,uncertain,REAL
1,R3W,0.439323,benign,REAL
2,R3M,0.399057,benign,REAL
3,S4L,0.585684,uncertain,REAL
4,P5S,0.569334,uncertain,REAL
5,P5R,0.643677,uncertain,REAL
6,P5L,0.645004,uncertain,REAL
7,L6V,0.522222,uncertain,REAL
8,K8R,0.525494,uncertain,REAL
9,A9V,0.517085,uncertain,REAL


## 3 · Where the REAL PrimateAI data came from

Covered right after Section 1 (before `load_primateai()` was first called): this
toolkit pulls PrimateAI's score out of **dbNSFP v5.0a** rather than PrimateAI's
own gated genome-wide release — see the build cell above for the exact steps,
license, and coverage caveat. Keyed by **genomic coordinate and protein_variant**;
CFTR is on the **plus** strand, so the coding alleles and the genomic `ref`/`alt`
are the same — no complementing needed.

In [5]:
info = tk.TOOL_REGISTRY['PrimateAI']
for key, val in info.items():
    print(f'  {key:12s}: {val}')

  kind        : missense
  learning    : semi-supervised
  signal      : deep net trained on common human/primate missense as a benign proxy
  circularity : medium
  pmid        : 30038395


## Key takeaways

1. **PrimateAI** is **semi-supervised** (benignity learned from common human/primate variants) → **medium** circularity. Cut `>= 0.803`.
2. Now **REAL** — but a **dbNSFP ClinVar subset** (~1,976 observed CFTR variants), not saturation. Honestly labelled; covers ~half of A1's observed set.
3. Non-commercial (dbNSFP CC BY-NC-ND) — raw parquet kept external.

**Next:** benchmark/00 — **ClinVar**.